# Data preprocess

In [ ]:
from pathlib import Path
import numpy as np
import snapatac2 as snap
from gcell._settings import get_setting
import scanpy as sc
import pandas as pd
from pyranges import PyRanges as pr
from preprocess_utils import (
    add_atpm,
    add_exp,
    create_peak_motif,
    download_motif,
    get_motif,
    query_motif,
)
import logging

# Configuration log
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  
        logging.FileHandler('process.log')  
    ]
)
logger = logging.getLogger(__name__)

# Get annotation_dir 
annotation_dir = get_setting('annotation_dir')
logger.info(f"gcell currently using annotation directory: {annotation_dir}")

# # read ATAC data
# logger.info("Loading ATAC data from ../seacells/process/atac.h5ad")
# ad_atac = sc.read_h5ad('../seacells/process/atac.h5ad')
# ad_atac.obs['cell_type'] = ad_atac.obs_names
# logger.info(f"ATAC data loaded with {ad_atac.n_obs} cells and {ad_atac.n_vars} peaks")

# # get peak name
# def get_peak_from_snapatac(atac: snap.AnnData):
#     logger.info("Extracting peak names from ATAC data")
#     peak_names = pd.DataFrame(atac.var.index.str.split('-').tolist(), columns=['Chromosome', 'Start', 'End'])
#     peak_names['Start'] = peak_names['Start'].astype(int)
#     peak_names['End'] = peak_names['End'].astype(int)
#     logger.info(f"Extracted {peak_names.shape[0]} peaks")
#     return peak_names

# peaks = get_peak_from_snapatac(ad_atac)
# logger.info(f"Peak dataframe shape: {peaks.shape}")

# # Get a specific cell type peak ACPM
# def get_peak_acpm_for_cell_type(atac: snap.AnnData, cell_type: str):
#     logger.info(f"Processing peaks for cell type: {cell_type}")
#     peaks = get_peak_from_snapatac(atac)
#     counts = np.array(atac[atac.obs.cell_type == cell_type].X.sum(0)).flatten()
#     acpm = np.log10(counts / counts.sum() * 1e5 + 1)
#     peaks['aCPM'] = acpm / acpm.max()
#     peaks = peaks.query('Chromosome.str.startswith("chr") & ~Chromosome.str.endswith("M") & ~Chromosome.str.endswith("Y") & ~Chromosome.str.startswith("chrUn")')
#     peaks = pr(peaks, int64=True).sort().df
#     logger.info(f"Processed {peaks.shape[0]} peaks for {cell_type}")
#     return peaks

# # Generate peak files for each cell type
# logger.info("Generating peak files for all cell types")
# for cell_type in ad_atac.obs['cell_type']:
#     peaks = get_peak_acpm_for_cell_type(ad_atac, cell_type)
#     output_file = f'./process_seacells/atac/{cell_type.replace(" ", "_").replace("/", "_").lower()}.atac.bed'
#     peaks.to_csv(output_file, sep='\t', index=False, header=False)
#     logger.info(f"Saved peaks to {output_file}")

# Read RNA data
logger.info("Loading RNA data from ../seacells/process/rna.h5ad")
rna = sc.read_h5ad('../seacells/process/rna.h5ad')
rna.obs['cell_type'] = rna.obs_names
logger.info(f"RNA data loaded with {rna.n_obs} cells and {rna.n_vars} genes")

# # Obtaining RNA TPMs for specific cell types
# def get_rna_for_cell_type(rna: snap.AnnData, cell_type: str):
#     logger.info(f"Processing RNA for cell type: {cell_type}")
#     counts = rna[rna.obs.cell_type == cell_type].X.sum(0)
#     counts = np.log10(counts / counts.sum() * 1e6 + 1)
#     counts = np.array(counts).flatten()
#     rna_tpm = pd.DataFrame(counts, columns=['TPM'])
#     rna_tpm['gene_name'] = rna.var.index
#     sorted_tpm = rna_tpm[['gene_name', 'TPM']].sort_values(by='gene_name', ascending=True)
#     logger.info(f"Processed RNA TPM for {cell_type}, shape: {sorted_tpm.shape}")
#     return sorted_tpm

# # Generate RNA files for each cell type
# logger.info("Generating RNA files for all cell types")
# for cell_type in rna.obs['cell_type']:
#     rna_tpm = get_rna_for_cell_type(rna, cell_type)
#     output_file = f'./process_seacells/rna/{cell_type.replace(" ", "_").replace("/", "_").lower()}.rna.csv'
#     rna_tpm.to_csv(output_file, index=False)
#     logger.info(f"Saved RNA TPM to {output_file}")

# built peak-motif data
logger.info("Creating peak-motif data")
peak_bed = "./process_seacells/atac/10x_datasets_1-1@108.atac.bed"
get_motif_output = './process_seacells/get_motif.bed'
create_peak_motif(get_motif_output, "./process_seacells/seacells.zarr", peak_bed)
logger.info("Peak-motif data created in ./process_seacells/seacells.zarr")

# daa ATPM data
logger.info("Adding ATPM data for all cell types")
for cell_type in rna.obs['cell_type']:
    cell_type_key = cell_type.replace(" ", "_").replace("/", "_").lower()
    atac_file = f"./process_seacells/atac/{cell_type_key}.atac.bed"
    add_atpm("./process_seacells/seacells.zarr", atac_file, cell_type_key)
    logger.info(f"Added ATPM for {cell_type_key} from {atac_file}")

# add expression data
logger.info("Adding expression data for all cell types")
for cell_type in rna.obs['cell_type']:
    cell_type_key = cell_type.replace(" ", "_").replace("/", "_").lower()
    rna_file = f"./process_seacells/rna/{cell_type_key}.rna.csv"
    atac_file = f"./process_seacells/atac/{cell_type_key}.atac.bed"
    add_exp(
        "./process_seacells/seacells.zarr",
        rna_file,
        atac_file,
        cell_type_key,
        assembly="hg38",
        version=44,
        extend_bp=300,
        id_or_name="gene_name"
    )
    logger.info(f"Added expression data for {cell_type_key} from {rna_file} and {atac_file}")

# Fine tuning

In [ ]:
from pathlib import Path
import numpy as np
import os
import logging
from get_model.config.config import load_config_from_yaml
from get_model.run_region import run_zarr as run
from pytorch_lightning import Trainer
import torch

# os.environ["WANDB_API_KEY"] = 'YOUR_API_KEY'
# os.environ["WANDB_MODE"] = "offline"

# Configure logging for debugging and monitoring
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load the list of cells
cell = os.listdir('./process_seacells/seacells.zarr/atpm/')
del cell[1]
n_total = len(cell)
n_per_chunk = n_total // 20  # Base size for each chunk
remainder = n_total % 20     # Remainder cells assigned to the first few chunks

# Load configuration and set a unified output directory
cfg = load_config_from_yaml("./training_config.yaml")
cfg.run.project_name = 'all_training'  # Project name
cfg.training.warmup_epochs = 0         # Avoid learning rate scheduler issues
logger.info(f"Starting training, planned for 10 epochs, with 20 chunks per epoch")

# Train for 10 epochs
# Note: range(2, 3) implies running only the 3rd epoch (index 2). Adjust range(10) for full training.
for epoch in range(2, 3):
    # Set random seed for each epoch
    np.random.seed(epoch)
    torch.manual_seed(epoch)
    cfg.run.run_name = f'epoch_{epoch+1}_full'  # Set a unique run name for each epoch
    
    # Randomly shuffle the cell list for each epoch
    shuffled_cell = np.random.permutation(cell)
    logger.info(f"Epoch {epoch+1}/10 - Shuffling cell order using seed {epoch}")
    
    # Split into 20 chunks based on the shuffled cell list
    chunks = []
    start = 0
    for i in range(20):
        chunk_size = n_per_chunk + (1 if i < remainder else 0)
        end = start + chunk_size
        chunks.append(shuffled_cell[start:end].tolist())
        start = end
    
    # Train 20 chunks and record cell names for each chunk
    # Note: range(18, 20) implies running only the last 2 chunks. Adjust range(20) for full training.
    for i in range(18, 20):
        logger.info(f"Epoch {epoch+1}/10 - Training chunk {i+1}/20, containing {len(chunks[i])} cells: {', '.join(chunks[i])}")
        cfg.dataset.celltypes = ','.join(chunks[i])
        
        # Determine which checkpoint to load
        if epoch == 0 and i == 0:
            # First chunk of the first epoch: Load default pre-trained checkpoint
            logger.info(f"Loading default pre-trained checkpoint: {cfg.finetune.checkpoint}")
        elif i == 0:
            # First chunk of subsequent epochs: Load final.ckpt from the previous epoch
            prev_final_checkpoint = f"{cfg.machine.output_dir}/{cfg.run.project_name}/epoch_{epoch}_full/checkpoints/final.ckpt"
            cfg.finetune.checkpoint = prev_final_checkpoint
            logger.info(f"Loading final checkpoint from previous epoch: {prev_final_checkpoint}")
        else:
            # Subsequent chunks within the same epoch: Load the latest versioned last.ckpt
            checkpoint_dir = f"{cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}/checkpoints"
            checkpoint_files = list(Path(checkpoint_dir).glob("last*.ckpt"))
            if checkpoint_files:
                latest_checkpoint = max(checkpoint_files, key=os.path.getctime)
                cfg.finetune.checkpoint = str(latest_checkpoint)
                logger.info(f"Loading latest versioned checkpoint: {cfg.finetune.checkpoint}")
            else:
                logger.warning("No versioned checkpoint found, continuing training with last known state")
        
        # Train the current chunk
        trainer = run(cfg)

    # Save the final checkpoint for the current epoch
    final_checkpoint = f"{cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}/checkpoints/final.ckpt"
    Path(final_checkpoint).parent.mkdir(parents=True, exist_ok=True)  # Ensure directory exists
    trainer.save_checkpoint(final_checkpoint)
    logger.info(f"Epoch {epoch+1}/10 completed, final checkpoint saved to {final_checkpoint}")
    
logger.info("Training for all 10 epochs completed")

# Performance evaluation

In [ ]:
import scanpy as sc
import random
import pandas as pd

## Data input

In [ ]:
adata = sc.read_h5ad('/path/to/seacells_project/process/rna.h5ad')
adata

In [ ]:
len(pd.DataFrame(adata.obs['celltype'].value_counts()))

In [ ]:
# Define the number of samples to be sampled for each cell type
n_samples = 3

# Use groupby().apply() to operate on each group
# lambda x: ... defines an anonymous function, where x represents the sub-DataFrame grouped by each cell type
# We check the number of rows in each group (len(x)), if it is greater than or equal to n_samples, randomly sample n_samples
# Otherwise (if the total number of cell types is less than n_samples), return the entire contents of the group
sampled_obs = adata.obs.groupby('celltype', group_keys=False).apply(
    lambda x: x.sample(n=n_samples) if len(x) >= n_samples else x
)

# Extract the row name (index) from the sampling result and convert it to a list
sampled_cell_list = sampled_obs.index.tolist()

In [ ]:
sampled_cell_list

In [ ]:
target_cell_types = ['HSC/MPP', 'ELP', 'CMP', 'MEMP']

hspc = adata[adata.obs['celltype'].isin(target_cell_types), :]
hspc

In [ ]:
hspc.obs_names

In [ ]:
# Define the number of samples to be sampled for each cell type
n_samples = 3

# Use groupby().apply() to operate on each group
# lambda x: ... defines an anonymous function, where x represents the sub-DataFrame grouped by each cell type
# We check the number of rows in each group (len(x)), if it is greater than or equal to n_samples, randomly sample n_samples
# Otherwise (if the total number of cell types is less than n_samples), return the entire contents of the group
sampled_obs = hspc.obs.groupby('celltype', group_keys=False).apply(
    lambda x: x.sample(n=n_samples) if len(x) >= n_samples else x
)

# Extract the row name (index) from the sampling result and convert it to a list
sampled_cell_list = sampled_obs.index.tolist()

In [ ]:
sampled_cell_list

## Data output

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from gcell.utils.causal_lib import get_subnet, plot_comm, preprocess_net

from get_model.config.config import export_config, load_config_from_yaml
from get_model.run_region import run_zarr as run

In [ ]:
from gcell.cell.celltype import GETHydraCellType

from get_model.config.config import load_config

### all

In [ ]:
adata = sc.read_h5ad('/path/to/seacells_project/process/rna.h5ad')

In [ ]:
cfg = load_config_from_yaml("training_config-tf.yaml")
use_checkpoint = '/path/to/get_project/process_seacells/all_training/epoch_3_full/checkpoints/best-v20.ckpt'
cfg.stage = 'predict'
cfg.dataset.leave_out_chromosomes = None
cfg.finetune.resume_ckpt = use_checkpoint
cfg.run.use_wandb=False
cfg.task.layer_names = []
cfg.task.gene_list = None
cfg.run.run_name='all'
cfg.dataset.leave_out_celltypes = celltype
cfg.dataset.zarr_path = '/path/to/get_project/process_seacells/seacells.zarr'
cfg.dataset.celltypes = ",".join([s.lower() for s in adata.obs_names.tolist()])

In [ ]:
for celltype in [
 'weng_nature_2024_2-2@112',
 'weng_nature_2024_2-1@37',
 'weng_nature_2024_2-1@84',
 '10x_datasets_6-1@8',
 'wimmers_cell_2023-61_cc0216@82',
 'wimmers_cell_2023-32_cc0022@65',
 'weng_nature_2024_4-3@16',
 'weng_nature_2024_3-3@38',
 'weng_nature_2024_3-3@64',
 'boukhaled_nat_immunol_2022-p3@22',
 'cheong_cell_2023_1-lgtd17@76',
 'wimmers_cell_2023-71_cc0594@22',
 'wimmers_cell_2023-47_cc0138@10',
 'wimmers_cell_2023-48_cc0138@100',
 'cheong_cell_2023_1-jcov72_5@81',
 '10x_datasets_2-1@64',
 'cheong_cell_2023_1-jcov111@30',
 'wimmers_cell_2023-60_cc0256@90',
 'boukhaled_nat_immunol_2022-hd2@51',
 'boukhaled_nat_immunol_2022-p3@133',
 'wimmers_cell_2023-60_cc0256@72',
 'cheong_cell_2023_1-jcov127@6',
 'li_nat_biotechnol_2023-mv_2@55',
 'neurips_2021-site1_donor2@73',
 'marderstein_nature_2024-ts21_16_a@30',
 'weng_nature_2024_4-3@52',
 'cheong_cell_2023_2-bmmcpie@39',
 'li_nat_biotechnol_2023-mv_1@2',
 'marderstein_nature_2024-ts21_11_g@25',
 'li_nat_biotechnol_2023-mv_2@74',
 'marderstein_nature_2024-ts21_11_g@76',
 'marderstein_nature_2024-ts21_11_g@16',
 'marderstein_nature_2024-ts21_16_a@67',
 'marderstein_nature_2024-ts21_11_g@0',
 'thomson_nat_immunol_2024-b076_p1_pb00599_02@271',
 'oguchi_science_2024-d2_cd4bulk_2@48',
 'wimmers_cell_2023-73_cc0710@95',
 'li_nat_biotechnol_2023-mv_1@45',
 'li_nat_biotechnol_2023-mv_2@56',
 'cheong_cell_2023_1-jcov72_5@6',
 'wimmers_cell_2023-51_cc0191@46',
 'weng_nature_2024_6-bmmc@134',
 'wimmers_cell_2023-64_cc0294@111',
 'mimitou_nat_biotechnol_2021-dig_stim@43',
 'mimitou_nat_biotechnol_2021-dig_stim@117',
 'mimitou_nat_biotechnol_2021-lll_ctrl@48',
 'weng_nature_2024_6-bmmc@26',
 'neurips_2021-site4_donor8@6',
 'weng_nature_2024_4-2@109',
 'li_nat_biotechnol_2023-mv_2@112',
 'li_nat_biotechnol_2023-mv_2@21',
 'li_nat_biotechnol_2023-mv_2@17',
 'li_nat_biotechnol_2023-mv_2@79',
 'li_nat_biotechnol_2023-mv_2@90',
 'cheong_cell_2023_1-jcov69_8@48',
 'neurips_2021-site1_donor1@24',
 '10x_datasets_1-1@55',
 'boukhaled_nat_immunol_2022-p4@51',
 'weng_nature_2024_6-bmmc@114',
 'weng_nature_2024_6-bmmc@100',
 'neurips_2021-site1_donor2@41',
 'weng_nature_2024_1-2@15',
 'boukhaled_nat_immunol_2022-p2@30',
 'weng_nature_2024_1-2@46',
 'boukhaled_nat_immunol_2022-p8@58',
 'cheong_cell_2023_1-hdv@69',
 'boukhaled_nat_immunol_2022-p4@34',
 '10x_datasets_1-1@81',
 'cheong_cell_2023_1-jcov72_5@34',
 'wimmers_cell_2023-37_cc0120@78',
 'weng_nature_2024_6-bmmc@120',
 'weng_nature_2024_6-bmmc@141',
 'neurips_2021-site1_donor2@21',
 'wimmers_cell_2023-66_cc0350@49',
 'wimmers_cell_2023-31_cc0022@2',
 'thomson_nat_immunol_2024-b069_p1_pb00323_02@76',
 'thomson_nat_immunol_2024-b076_p1_pb00124_02@17',
 'weng_nature_2024_5-2@59',
 'weng_nature_2024_4-1@12',
 'weng_nature_2024_1-1@55',
 'thomson_nat_immunol_2024-b076_p1_pb00368_04@123',
 'wei_immuni_2023-yw8@80',
 'thomson_nat_immunol_2024-b076_p1_pb00599_02@317',
 'thomson_nat_immunol_2024-b076_p1_pb00124_02@68',
 'wimmers_cell_2023-39_cc0069@49',
 'thomson_nat_immunol_2024-b076_p1_imm19_445@21',
 'lareau_nat_genet_2023-dogma_pt1_rep2@45',
 'weng_nature_2024_2-2@31',
 'lareau_nat_genet_2023-dogma_pt1_rep1@120',
 'mimitou_nat_biotechnol_2021-dig_stim@41',
 'cheong_cell_2023_1-jcov72_5@84',
 'boukhaled_nat_immunol_2022-p4@57',
 '10x_datasets_4-1@27',
 'marderstein_nature_2024-ts21_16_b@28',
 'boukhaled_nat_immunol_2022-p6@22',
 'wimmers_cell_2023-35_cc0061@87',
 'thomson_nat_immunol_2024-b076_p1_pb00124_02@243',
 'thomson_nat_immunol_2024-b076_p1_pb00377_03@52',
 'cheong_cell_2023_1-jcov101@4',
 'wimmers_cell_2023-56_cc0249@72',
 'cheong_cell_2023_1-lgtd8m@76',
 'neurips_2021-site2_donor1@54'
]:
    zarr_path = './tf/tf/all/'+celltype+'.zarr'
    hydra_celltype = GETHydraCellType.from_config(cfg, 
                                              celltype=celltype, 
                                              zarr_path=zarr_path)
    hydra_celltype.gene_annot[hydra_celltype.gene_annot['Chromosome'] == 'chr11'].to_csv('./tf/evalu/all/'+celltype+'.csv')

### hspc

In [ ]:
cfg = load_config_from_yaml("training_config-tf.yaml")
use_checkpoint = '/path/to/get_project/process_seacells/finetune_hspc/finetune_hspc_1353_chr11/checkpoints/best-v1.ckpt'
cfg.stage = 'predict'
cfg.dataset.leave_out_chromosomes = None
cfg.finetune.resume_ckpt = use_checkpoint
cfg.run.use_wandb=False
cfg.task.layer_names = []
cfg.task.gene_list = None
cfg.run.run_name='hspc'
cfg.dataset.leave_out_celltypes = celltype
cfg.dataset.zarr_path = '/path/to/get_project/process_seacells/seacells.zarr'
cfg.dataset.celltypes = ",".join([s.lower() for s in hspc.obs_names.tolist()])

In [ ]:
for celltype in [
 'weng_nature_2024_4-2@39',
 'weng_nature_2024_5-3@126',
 'weng_nature_2024_4-3@33',
 'cheong_cell_2023_1-jcov101_2@45',
 'li_nat_biotechnol_2023-mv_1@11',
 'weng_nature_2024_5-2@10',
 'marderstein_nature_2024-ts21_11_g@44',
 'cheong_cell_2023_1-hd_ad@41',
 'weng_nature_2024_4-2@5',
 'li_nat_biotechnol_2023-mv_2@112',
 'li_nat_biotechnol_2023-mv_2@118',
 'li_nat_biotechnol_2023-mv_2@82'
]:
    zarr_path = './tf/tf/hspc/'+celltype+'.zarr'
    hydra_celltype = GETHydraCellType.from_config(cfg, 
                                              celltype=celltype, 
                                              zarr_path=zarr_path)
    hydra_celltype.gene_annot[hydra_celltype.gene_annot['Chromosome'] == 'chr11'].to_csv('./tf/evalu/hspc/'+celltype+'.csv')

## Data sorting

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score

# 1. Set the target directory
# Path('.') represents the current directory; modify to absolute or relative path as needed
base_dir = Path('./tf/evalu/all/')

# 2. Prepare a dictionary to store the final results
results_dict = {}

# 3. Check if the target directory exists
if not base_dir.is_dir():
    print(f"Error: Directory '{base_dir}' does not exist.")
else:
    # 4. Find and iterate through all .csv files in the directory
    csv_files = list(base_dir.glob('*.csv'))
    print(f"Found {len(csv_files)} CSV files in '{base_dir}', starting processing...")

    for file_path in csv_files:
        print(f"--- Processing: {file_path.name} ---")
        try:
            # Read the CSV file
            data = pd.read_csv(file_path)

            # Check if required columns exist
            required_columns = ['gene_name', 'pred', 'obs']
            if not all(col in data.columns for col in required_columns):
                print(f"  -> Warning: File {file_path.name} is missing required columns, skipped.")
                continue

            # Perform the specified core operations: Sort and Drop Duplicates
            processed_data = data.sort_values(
                by=['pred', 'obs'], 
                ascending=[False, False]
            ).drop_duplicates(
                subset=['gene_name'], 
                keep='first'
            )

            # Check if processed data is sufficient for calculation (at least 2 rows)
            if len(processed_data) < 2:
                print(f"  -> Warning: File {file_path.name} has fewer than 2 rows after processing, cannot calculate correlations, skipped.")
                continue

            # Calculate three metrics
            # a. Pearson Correlation
            pearson_corr = processed_data['obs'].corr(processed_data['pred'], method='pearson')
            
            # b. Spearman Rank Correlation
            spearman_corr = processed_data['obs'].corr(processed_data['pred'], method='spearman')
            
            # c. R-squared (Coefficient of Determination)
            r_squared = r2_score(processed_data['obs'], processed_data['pred'])
            
            # Store results in dictionary, using filename (without extension) as key
            file_name_stem = file_path.stem
            results_dict[file_name_stem] = [pearson_corr, spearman_corr, r_squared]
            
            print(f"  -> Processed successfully. Pearson: {pearson_corr:.4f}, Spearman: {spearman_corr:.4f}, R-squared: {r_squared:.4f}")

        except Exception as e:
            # Catch other possible errors, such as file reading failures
            print(f"  -> Error occurred while processing file {file_path.name}: {e}")

    # 5. Convert the results dictionary to the final DataFrame
    if results_dict:
        final_df = pd.DataFrame.from_dict(
            results_dict, 
            orient='index', 
            columns=['Pearson_Correlation', 'Spearman_Correlation', 'R_Squared']
        )
        
        print("\n\n--- All files processed ---")
        print("Final Results Summary:")
        print(final_df)
        
        # (Optional) Save results to a CSV file
        # final_df.to_csv('analysis_summary.csv')
        # print("\nResults saved to analysis_summary.csv")

    else:
        print("\nNo files were successfully processed.")

In [ ]:
final_df

In [ ]:
final_df.to_csv('./tf/evalu/all.csv')

## Visual

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# --- 1. Setup Parameters ---
# Path to the directory containing data
data_directory = Path('./tf/evalu/')
# Note: This script now automatically identifies all numeric columns in the CSVs, no need to specify manually
# DATA_COLUMNS = ['column_1', 'column_2', 'column_3'] 

# --- 2. Read and Consolidate all CSV Files ---
all_data_frames = []
csv_files = list(data_directory.glob('*.csv'))

if not csv_files:
    print(f"Error: No CSV files found in directory '{data_directory}'.")
else:
    print(f"Found {len(csv_files)} CSV files, starting processing...")
    for file_path in csv_files:
        try:
            # Read CSV file
            df = pd.read_csv(file_path, index_col=0)
            
            # Add a new column to identify the data source (filename)
            df['source_file'] = file_path.stem 
            all_data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")

# Concatenate all data into a single DataFrame
if all_data_frames:
    combined_df = pd.concat(all_data_frames, ignore_index=True)

    # --- 3. Reshape Data (Melt) ---
    # Convert wide-format data to long-format for plotting
    actual_data_columns = [col for col in combined_df.columns if col != 'source_file']
    
    long_form_df = combined_df.melt(
        id_vars=['source_file'],
        value_vars=actual_data_columns,
        var_name='data_column', # New column name for original column headers
        value_name='value'      # New column name for values
    )

    # --- 4. Plot Boxplot using Seaborn ---
    print("Data prepared, starting plotting...")
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(14, 8))

    # --- Main modification here: Swapped x and hue values ---
    # Now x-axis is source file, hue (color/grouping) is data column
    sns.boxplot(
        data=long_form_df,
        x='source_file',
        y='value',
        hue='data_column'
    )
    # ---------------------------------------------

    # --- 5. Customize Chart Aesthetics ---
    plt.title('Comparison of Data Columns Distribution Across File Sources', fontsize=16)
    plt.xlabel('File Source', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.xticks(rotation=15)
    plt.legend(title='Data Column') # Legend title is now "Data Column"
    plt.tight_layout()

    # (Optional) Save chart to file
    output_filename = 'boxplot_comparison.pdf'
    plt.savefig(output_filename, dpi=300)
    print(f"Chart saved as {output_filename}")
    
    # Show chart
    plt.show()

# Interpretation

## Compute

In [ ]:
import logging  
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from gcell.utils.causal_lib import get_subnet, plot_comm, preprocess_net

from get_model.config.config import export_config, load_config_from_yaml, load_config
from get_model.run_region import run_zarr as run

from gcell.cell.celltype import GETHydraCellType

import scanpy as sc
from gcell.cell.celltype import parallel_get_gene_by_motif

i = 0
for celltype in ['b_cells','b-cell_lineage','cycling_cells','dc','early_mk','erythroid','hsc_mpp','ilc','macrophages','mast_cells','monocytes','pdc','plasma_cells','t_cells']:
    i += 1
    try:
        cfg = load_config_from_yaml("training_config-tf.yaml")
        # use_checkpoint = f'{cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}/checkpoints/best-v1.ckpt' 
        use_checkpoint = '/path/to/get_project/process_seacells/all_training/epoch_3_full/checkpoints/best-v20.ckpt'
        cfg.stage = 'predict'
        cfg.dataset.leave_out_chromosomes = None
        cfg.finetune.resume_ckpt = use_checkpoint
        cfg.run.use_wandb=False
        cfg.task.layer_names = []
        cfg.task.gene_list = None
        cfg.run.run_name='tf'
        cfg.dataset.leave_out_celltypes = celltype
        # trainer = run(cfg)
        
        zarr_path = './tf/tf/tf/'+celltype+'.zarr'
        hydra_celltype = GETHydraCellType.from_config(cfg, celltype=celltype, zarr_path=zarr_path)
        
        parallel_get_gene_by_motif(zarr_path, max_workers = 20)
        
        hydra_celltype.gene_by_motif.get_causal()
        
    except Exception as e:
        continue  

## Analysis

### motif search

In [ ]:
hydra_celltype.motif.annotations[hydra_celltype.motif.annotations['Name'] == 'ZNF5']['Motif'].tolist()

In [ ]:
## RUNX1
# EVI1/MECOM: MECOM
# ZNF140: ZNF140
# TBX/3: TBX3, TBX21, TBX2, EOMES
# HD/22: NKX3-1, NKX2-1, NKX2-5, NKX2-3, NKX2-8
# NR/19: NR1D1, NR1D2, RORA, RORB, RORC
# NFKB/1: NFKB1, NFKB2, REL, RELA, RELB

In [ ]:
## GFI1B
# 'CCAAT/CEBP': CEBPA, CEBPB, CEBPD, CEBPE, CEBPG, DBP, HLF, TEF, NFIL3
# 'NFAT/1': NFATC1, NFATC2, NFATC3, NFATC4
# 'NR/1': HNF4A, HNF4G, NR1H2, NR1H3, NR2C2, NR2F1, NR2F6, RXRA, RXRB, RXRG
# 'ZNF134': ZNF134
# 'ZNF418': ZNF418

In [ ]:
## GATA2
# 'HIC/1': HIC1, HIC2
# 'LEF1': LEF1, TCF7, ZNF350
# 'NR/11': ESRRB, NR5A2, NR6A1, RXRA, RARA, RXRG
# 'PRDM16': PRDM16
# 'ZNF143': IKZF1, ZNF76, ZNF143

In [ ]:
## EVI1/MECOM
# 'EBF1':  EBF1
# 'MEF2':  MEF2A, MEF2B, MEF2C, MEF2D
# 'NR/11':  ESRRB, NR5A2, NR6A1, RXRA, RARA, RXRG
# 'RBPJ':  RBPJ, ZNF75A
# 'SRF': SRF

In [ ]:
## ZNF521


In [ ]:
hydra_celltype.motif.annotations[hydra_celltype.motif.annotations['Name'] == 'ZNF5']

In [ ]:
hydra_celltype.motif.annotations.query('Motif.str.startswith("FOXG")')

In [ ]:
hydra_celltype.motif.annotations.query('Motif.str.startswith("TP53")')

In [ ]:
hydra_celltype.motif.annotations.to_csv('./process_seacells/motif_cluster.csv')

In [ ]:
hydra_celltype.motif.annotations['Name'].value_counts().to_csv('motif_number.csv')

### interaction network

In [ ]:
from gcell.utils.causal_lib import get_subnet, plot_comm, preprocess_net

In [ ]:
g = hydra_celltype.gene_by_motif.get_causal()
g_ = preprocess_net(g, 0.24)

In [ ]:
plot_comm(get_subnet(g_.copy(), 'EVI1/MECOM'), figsize=(10, 10))

In [ ]:
nodes = sorted(list(get_subnet(g_.copy(), 'EVI1/MECOM').nodes()), key=lambda x: str(x))
nodes

In [ ]:
plot_comm(g_, figsize=(20, 20))

In [ ]:
nodes = list(g_.nodes())
nodes

In [ ]:
import networkx as nx
import pickle

# Let's assume 'g' is your graph

# Save the graph object using pickle
with open("my_graph.pickle", "wb") as f:
    pickle.dump(g, f)

print("Graph successfully saved to my_graph.pickle")

### network visual

In [ ]:
def get_color(i, r_off=1, g_off=1, b_off=1):
    """Assign a color to a vertex.

    Parameters
    ----------
    i: int
        The index of the vertex.
    r_off: int, optional
        The offset for the red channel, defaults to 1.
    g_off: int, optional
        The offset for the green channel, defaults to 1.
    b_off: int, optional
        The offset for the blue channel, defaults to 1.
    """
    n = 16
    low, high = 0.5, 0.9
    span = high - low
    r = low + span * (((i + r_off) * 3) % n) / (n - 1)
    g = low + span * (((i + g_off) * 5) % n) / (n - 1)
    b = low + span * (((i + b_off) * 7) % n) / (n - 1)
    return (r, g, b)


In [ ]:
def plot_comm(G, figsize=(10, 10), title="Network structure", savefig=False, edge_scale_factor=1.5):
    """Plot the community structure of the graph.

    Parameters
    ----------
    G: nx.Graph
        The graph to plot.
    figsize: tuple, optional
        The size of the figure, defaults to (10, 10).
    title: str, optional
        The title of the plot, defaults to "Network structure".
    savefig: str, optional
        The path to save the plot, defaults to False.
    edge_scale_factor: float, optional
        The factor to scale edge widths, defaults to 1.5.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    # Set community color for internal edges
    external = [(v, w) for v, w in G.edges if G.edges[v, w]["community"] == 0]
    internal = [(v, w) for v, w in G.edges if G.edges[v, w]["community"] > 0]
    internal_color = [
        "#fddbc7" if (G.edges[e]["weight"] > 0) else "#d1e5f0" for e in internal
    ]
    external_color = [
        "#ef8a62" if (G.edges[e]["weight"] > 0) else "#67a9cf" for e in external
    ]
    node_color = [get_color(G.nodes[v]["community"]) for v in G.nodes]

    weights = [np.absolute(G[u][v]["weight"]) * edge_scale_factor for u, v in G.edges]  # Scale edge widths
    # external edges
    pos = nx.spring_layout(G)
    nx.draw(
        G,
        pos=pos,
        node_size=0,
        edgelist=external,
        width=weights,
        ax=ax,
        edge_color=external_color,
        node_color=node_color,
        with_labels=True,
    )
    # internal edges
    nx.draw(
        G,
        pos=pos,
        node_size=100,
        edgelist=internal,
        ax=ax,
        width=weights,
        edge_color=internal_color,
        node_color=node_color,
        with_labels=True,
    )
    ax.set_title(title)
    plt.tight_layout()
    # add padding in the margins
    plt.margins(0.3, 0.3)
    if savefig:
        save_path = Path(savefig) / f"{title}.pdf"
        plt.savefig(save_path, dpi=300)
    plt.show()
    plt.close()

In [ ]:
import numpy as np
import networkx as nx

In [ ]:
plot_comm(g_, figsize=(20, 20), savefig='./',edge_scale_factor=1)

In [ ]:
plot_comm(get_subnet(g_.copy(), 'EVI1/MECOM'), figsize=(10, 10), savefig='./', edge_scale_factor=5)

In [ ]:
plot_comm(get_subnet(g_.copy(), 'RUNX/1'), figsize=(10, 10), savefig='./', edge_scale_factor=5)

In [ ]:
plot_comm(g_, figsize=(20, 20), savefig='./', edge_scale_factor=2)

### Gene annotation and correlation calculation

In [ ]:
hydra_celltype.gene_annot

In [ ]:
pearson_corr = hydra_celltype.gene_annot[['pred', 'obs']].corr(method='pearson')
print("Pearson:\n", pearson_corr)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=hydra_celltype.gene_annot,
    x='pred',
    color='blue',
    label='Predicted',
    kde=True,
    alpha=0.3
)

sns.histplot(
    data=hydra_celltype.gene_annot,
    x='obs',
    color='orange',
    label='Observed',
    kde=True,
    alpha=0.3
)

plt.title("Distribution Comparison")
plt.xlabel("Value")
plt.legend()
plt.show()

### motif-gene

In [ ]:
import pandas as pd

In [ ]:
?hydra_celltype.gene_jacobian_summary

In [ ]:
data = pd.DataFrame(hydra_celltype.gene_jacobian_summary('BCL11A', axis='motif')).sort_values(by=0, ascending=False)
data

#### motif-gene link

In [ ]:
hydra_celltype.gene_by_motif.data

In [ ]:
hydra_celltype.gene_by_motif.data.to_csv('./process_seacells/bulk_hsc_mpp_motif_gene.csv')

In [ ]:
hydra_celltype.gene_by_motif.data.loc['MECOM',]

In [ ]:
hydra_celltype.gene_by_motif.data[['EVI1/MECOM']].sort_values(by='EVI1/MECOM', ascending=False).to_csv('./process_seacells/MECOM_socre.csv')

#### motif-gene visual

In [ ]:
?hydra_celltype.plot_gene_motifs

In [ ]:
hydra_celltype.motif.annotations.query('Motif.str.startswith("GATA")')

In [ ]:
hydra_celltype.gene_by_motif.data.loc['BCL11A',:].sort_values(ascending=False)[:20]

In [ ]:
hydra_celltype.plot_gene_motifs('MECOM', n=11)

### peak-gene

In [ ]:
len(hydra_celltype.peak_annot['Gene'].unique().tolist())

In [ ]:
hydra_celltype.peak_annot['Gene'].unique().tolist()

In [ ]:
data = pd.DataFrame(hydra_celltype.gene_jacobian_summary('GNAS', axis='region')).sort_values(by='Score', ascending=False)
data

In [ ]:
import pandas as pd

# Get a list of unique genes
gene_list = hydra_celltype.peak_annot['Gene'].unique().tolist()

# Initialize an empty list to store the results for each gene
df_list = []
i = 0

# Iterate through each gene
for gene in gene_list:
    i = i + 1
    print(i)
    # Extract Jacobian summary data and convert it to a DataFrame
    data = pd.DataFrame(hydra_celltype.gene_jacobian_summary(gene, axis='region'))
    
    # Sort the data by 'Score' in descending order
    data = data.sort_values(by='Score', ascending=False)
    
    # Keep only rows where Score > 0
    data = data[data['Score'] > 0]
    
    # Add a 'Gene' column
    data['Gene'] = gene
    
    # Append the result to the list
    df_list.append(data)

In [ ]:
final_df = pd.concat(df_list, axis=0, ignore_index=True)

In [ ]:
print(final_df)

In [ ]:
final_df.to_csv('peak_gene.csv')